In [4]:
import csv
from web3 import Web3
from eth_abi import decode
import pandas as pd
from tqdm import tqdm

event_signatures = {
    'OrderFilled': 'OrderFilled(bytes32,address,address,uint256,uint256,uint256,uint256,uint256)',
}

event_topic0 = {name: Web3.keccak(text=signature).hex() for name, signature in event_signatures.items()}
print(event_topic0)

{'OrderFilled': 'd0a08e8c493f9c94f29311604c9de1b4e8c8d4c06bd0c789af57f2d65bfec0f6'}


In [20]:

print('Reading logs...')

def decode_address(topic):
    return '0x' + topic[-40:]

def decode_uint256(data):
    return int(data, 16)

def decode_order_filled(log):
    decoded_event = {}
    # More for debugging
    decoded_event['event'] = 'OrderFilled'
    decoded_event['blockNumber'] = decode_uint256(log['blockNumber'])
    decoded_event['txIndex'] = decode_uint256(log['txIndex'])

    # Topic 1
    decoded_event['orderHash'] = log['topic1']
    
    # Topic 2
    decoded_event['maker'] = decode_address(log['topic2'])
    
    # Topic 3
    taker = decode_address(log['topic3'])
    if (taker == '0x4bfb41d5b3570defd03c39a9a4d8de6bd8b8982e'):
        decoded_event['taker'] = 'CTF_Exchange'
    else:
        decoded_event['taker'] = taker
    
    # Data Field: /wrong
    assert(log['data'].startswith('0x'))
    data = log['data'][2:]
    

    decoded_event['makerAssetId'] = decode_uint256(data[0:64])
    decoded_event['takerAssetId'] = decode_uint256(data[64:128])
    decoded_event['makerAmountFilled'] = decode_uint256(data[128:192])
    decoded_event['takerAmountFilled'] = decode_uint256(data[192:256])
    decoded_event['fee'] = decode_uint256(data[256:320])
    # Decoding
    # decoded_event['data'] = data
    
    return decoded_event

# Read log entries
total_log_entries = 0
decoded_events = []
non_decoded_events = []
non_decoded_events_log_index = [] 

# Choose starting and ending log file
start = 35
end = 59



for i in tqdm(range(start, end), colour='green', ncols=(end - start)*3):
    log_entries = []

        
    with open(f'ctf_exchange/logs_{i}M.csv', 'r') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            log_entries.append(row)


    for log in log_entries:
        topic0 = log['topic0']
        total_log_entries += 1
        if topic0 == '0x' + event_topic0['OrderFilled']:
            decoded_event = decode_order_filled(log)
            decoded_events.append(decoded_event)

print('Decoded events:', len(decoded_events))

Reading logs...


100%|███████████████████████████████████| 24/24 [00:09<00:00,  2.46it/s]

Decoded events: 913928


In [21]:
df = pd.DataFrame(decoded_events)
df.to_csv(f'orderfilled_events/orderfilled_{start}-{end}.csv', index=False)

In [22]:
df_no_ctfexchange = df.drop(df[df['taker'] == 'CTF_Exchange'].index, inplace = True)
df.to_csv(f'orderfilled_events/orderfilled_{start}-{end}_no_ctfexchange.csv', index=False)